In [1]:
# ============================================================
# CELL 1 — Install dependencies
# ============================================================
# !pip -q install -U pandas numpy pyarrow unidecode tqdm matplotlib seaborn

In [2]:
# ============================================================
# CELL 2 — Imports, paths, config
# ============================================================
from pathlib import Path
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from unidecode import unidecode

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)

# -------------------------------------------------------------------
# PATHS — adjust ROOT if your folder differs
# -------------------------------------------------------------------
ROOT_CANDIDATES = [Path("/root/dataset"), Path("root/dataset"), Path("dataset")]
ROOT = next((p for p in ROOT_CANDIDATES if p.exists()), None)
assert ROOT is not None, "Dataset folder not found."

NMAT_FINAL_PATH    = ROOT / "NMAT_FINAL.csv"
PLE_DATA_PATH      = ROOT / "PLE_DATA.csv"
PLE_UNMATCHED_PATH = ROOT / "PLE_UNMATCHED.csv"
PLE_STILL_UNMATCHED_PATH = ROOT / "output" / "PLE_STILL_UNMATCHED.csv"

OUTDIR = ROOT / "output"
OUTDIR.mkdir(parents=True, exist_ok=True)

ULTIMA_PATH = ROOT / "NMAT_Ultima.csv"

assert NMAT_FINAL_PATH.exists(),    f"Missing: {NMAT_FINAL_PATH}"
assert PLE_DATA_PATH.exists(),      f"Missing: {PLE_DATA_PATH}"
assert PLE_UNMATCHED_PATH.exists(), f"Missing: {PLE_UNMATCHED_PATH}"
# Note: PLE_STILL_UNMATCHED_PATH might not exist until the file is placed, so we won't assert it strictly initially

# -------------------------------------------------------------------
# MATCHING CONFIG
# -------------------------------------------------------------------
YEAR_GAP_MIN          = 5    # PLE_YEAR - NMAT_YEAR >= 5 (medical school takes ≥4 yrs + boards)
# PERCENTILE_FLOOR is intentionally UNUSED in disambiguate() as of this fix
# (ORCHESTRATOR RULING). It used to hard-filter out, and even outright reject,
# PLE-name candidates scoring below the 40th percentile during IDENTITY
# resolution -- but percentile is exactly the outcome variable this project
# studies (does below-40th-percentile status predict PLE non-linkage?), so
# using it to decide WHO a PLE record belongs to baked part of the answer
# into the question. Kept here, unused, so the constant's history is visible.
PERCENTILE_FLOOR      = 40   # NO LONGER USED for identity resolution (see disambiguate())

print("ROOT:", ROOT)
print("NMAT_FINAL_PATH:", NMAT_FINAL_PATH)
print("PLE_DATA_PATH:", PLE_DATA_PATH)
print("PLE_UNMATCHED_PATH:", PLE_UNMATCHED_PATH)
print("YEAR_GAP_MIN:", YEAR_GAP_MIN)
print("PERCENTILE_FLOOR:", PERCENTILE_FLOOR)

ROOT: dataset
NMAT_FINAL_PATH: dataset\NMAT_FINAL.csv
PLE_DATA_PATH: dataset\PLE_DATA.csv
PLE_UNMATCHED_PATH: dataset\PLE_UNMATCHED.csv
YEAR_GAP_MIN: 5
PERCENTILE_FLOOR: 40


d:\User\Desktop\Acads\NMAT Analysis\NMAT_Analysis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# ============================================================
# CELL 3 — Name normalization helpers
# ============================================================

def normalize_name(x) -> str:
    """
    Normalize a Filipino name for matching.
    1. Strip accents (Añes → Anes)
    2. Uppercase
    3. Remove punctuation except commas and spaces
    4. Collapse whitespace
    5. Strip leading/trailing
    Returns empty string if null/empty.
    """
    if pd.isna(x):
        return ""
    x = str(x)
    x = unidecode(x)          # strip diacritics
    x = x.upper()
    x = re.sub(r"[^A-Z0-9, ]+", " ", x)   # keep letters, digits, comma, space
    x = re.sub(r"\s+", " ", x).strip()
    return x


def clean_appno(x) -> str:
    """Strip non-digits from applicant number."""
    if pd.isna(x):
        return ""
    s = re.sub(r"\D", "", str(x))
    return s if s else ""


def clean_year(x):
    """Convert year to int; return NaN if not parseable."""
    try:
        v = int(float(str(x).strip()))
        return v if 1990 <= v <= 2030 else np.nan
    except Exception:
        return np.nan


# Quick sanity tests
assert normalize_name("Añes, Marvin") == "ANES, MARVIN"
assert normalize_name("ABANDO, ALEJANDRO LUIS ARELLANO") == "ABANDO, ALEJANDRO LUIS ARELLANO"
assert clean_appno("1070637") == "1070637"
assert clean_appno(None) == ""

print("Name normalization helpers OK.")

Name normalization helpers OK.


In [4]:
# ============================================================
# CELL 4 — Load all three source files
# ============================================================

# ── NMAT_FINAL ────────────────────────────────────────────
nmat = pd.read_csv(NMAT_FINAL_PATH, dtype=str, low_memory=False)

num_cols = [
    "Year", "NMS_PER_num", "NMS_GPS", "NMS_APT", "NMS_SA",
    "NMS_VCss","NMS_IRss","NMS_Qss","NMS_PAss",
    "NMS_BIOss","NMS_PHYss","NMS_SSCss","NMS_CHEMss",
    "TotalRawScoreTRUE","PartIRawScoreTRUE","PartIIRawScoreTRUE",
    "Raw_Verbal","Raw_InductiveReasoning","Raw_Quantitative","Raw_PerceptualAcuity",
    "Raw_Biology","Raw_Physics","Raw_SocialScience","Raw_Chemistry",
    "StoredRawTotal","CalculatedRawTotal_Source",
    "APT_CEM","SA_CEM","GPS_CEM","Percentile_CEM"
]
for col in num_cols:
    if col in nmat.columns:
        nmat[col] = pd.to_numeric(nmat[col], errors="coerce")

# Normalize names and keys
nmat["NAME_NORM"]   = nmat["NMA_Name"].map(normalize_name)
nmat["APPNO_CLEAN"] = nmat["NMA_AppNo"].map(clean_appno)
nmat["YEAR_INT"]    = nmat["Year"].map(clean_year)

print(f"NMAT_FINAL loaded:     {len(nmat):,} rows  |  {nmat['APPNO_CLEAN'].nunique():,} unique AppNos")
print(f"  Name coverage:       {nmat['NAME_NORM'].ne('').sum():,} non-empty names")

# ── PLE_DATA ──────────────────────────────────────────────
ple = pd.read_csv(PLE_DATA_PATH, dtype=str)
ple.columns = [c.strip() for c in ple.columns]
ple["FULL_NAME"]       = ple["FULL_NAME"].str.strip()
ple["PLE_YEAR_PASSED"] = ple["PLE_YEAR_PASSED"].map(clean_year)
ple["NAME_NORM"]       = ple["FULL_NAME"].map(normalize_name)

print(f"\nPLE_DATA loaded:       {len(ple):,} rows")
print(f"  Year range:          {int(ple['PLE_YEAR_PASSED'].min())} – {int(ple['PLE_YEAR_PASSED'].max())}")
print(f"  Unique names:        {ple['NAME_NORM'].nunique():,}")

# ── PLE_UNMATCHED ─────────────────────────────────────────
unmatched = pd.read_csv(PLE_UNMATCHED_PATH, dtype=str)
unmatched.columns = [c.strip() for c in unmatched.columns]
unmatched["FULL_NAME"]  = unmatched["FULL_NAME"].str.strip()
unmatched["NMA_AppNo"]  = unmatched["NMA_AppNo"].fillna("").str.strip()
unmatched["NAME_NORM"]  = unmatched["FULL_NAME"].map(normalize_name)
unmatched["APPNO_CLEAN"]= unmatched["NMA_AppNo"].map(clean_appno)

has_appno = unmatched["APPNO_CLEAN"].ne("").sum()
print(f"\nPLE_UNMATCHED loaded:  {len(unmatched):,} rows")
print(f"  With AppNo filled:   {has_appno:,}")
print(f"  AppNo empty:         {len(unmatched) - has_appno:,}")

# ── PLE_STILL_UNMATCHED ───────────────────────────────────
ple_still_unmatched = pd.read_csv(PLE_STILL_UNMATCHED_PATH, dtype=str)
ple_still_unmatched.columns = [c.strip() for c in ple_still_unmatched.columns]

# BUGFIX (A2/orchestrator, 2026-08): this file is a fossil of an earlier
# pipeline version that DID run fuzzy name matching -- 4,810 of its 7,207 rows
# carry MATCH_METHOD="FUZZY". Stage 2 consumes its MATCHED_APPNO values and
# labels the result DETERMINISTIC_APPNO, which laundered fuzzy provenance into
# a stage the project advertises as deterministic-only and human-auditable.
# Only rows carrying a usable MATCHED_APPNO can contribute at all, so the real
# exposure is 269 fuzzy-derived AppNos (not 4,810) -- cheap to drop, and
# dropping them makes the "no fuzzy matching" guarantee true of the lineage and
# not merely of the current source code.
if "MATCH_METHOD" in ple_still_unmatched.columns:
    _fuzzy = ple_still_unmatched["MATCH_METHOD"].astype(str).str.upper().eq("FUZZY")
    _fuzzy_with_appno = int((_fuzzy & ple_still_unmatched["MATCHED_APPNO"]
                             .fillna("").str.strip().ne("")).sum())
    ple_still_unmatched = ple_still_unmatched[~_fuzzy].copy()
    print(f"PLE_STILL_UNMATCHED: dropped {int(_fuzzy.sum()):,} FUZZY-derived rows "
          f"({_fuzzy_with_appno:,} of which carried a usable AppNo) "
          f"-- deterministic-only guarantee now holds for the lineage too")
# Determine which AppNo column is present
appno_col = "NMA_AppNo" if "NMA_AppNo" in ple_still_unmatched.columns else "MATCHED_APPNO"
if appno_col in ple_still_unmatched.columns:
    ple_still_unmatched["NMA_AppNo"] = ple_still_unmatched[appno_col].fillna("").str.strip()
else:
    ple_still_unmatched["NMA_AppNo"] = ""
ple_still_unmatched["APPNO_CLEAN"] = ple_still_unmatched["NMA_AppNo"].map(clean_appno)
ple_still_unmatched["NAME_NORM"] = ple_still_unmatched["PLE_NAME_NORM"] # Usually it's PLE_NAME_NORM
print(f"\nPLE_STILL_UNMATCHED loaded:  {len(ple_still_unmatched):,} rows")
print(f"  With AppNo filled:         {ple_still_unmatched['APPNO_CLEAN'].ne('').sum():,}")

NMAT_FINAL loaded:     178,927 rows  |  178,926 unique AppNos
  Name coverage:       178,927 non-empty names

PLE_DATA loaded:       43,630 rows
  Year range:          2011 – 2022
  Unique names:        43,630

PLE_UNMATCHED loaded:  6,600 rows
  With AppNo filled:   2,332
  AppNo empty:         4,268
PLE_STILL_UNMATCHED: dropped 4,810 FUZZY-derived rows (269 of which carried a usable AppNo) -- deterministic-only guarantee now holds for the lineage too

PLE_STILL_UNMATCHED loaded:  2,397 rows
  With AppNo filled:         52


In [5]:
# ============================================================
# CELL 4 — Load all three source files
# ============================================================

# ── NMAT_FINAL ────────────────────────────────────────────
nmat = pd.read_csv(NMAT_FINAL_PATH, dtype=str, low_memory=False)

num_cols = [
    "Year", "NMS_PER_num", "NMS_GPS", "NMS_APT", "NMS_SA",
    "NMS_VCss","NMS_IRss","NMS_Qss","NMS_PAss",
    "NMS_BIOss","NMS_PHYss","NMS_SSCss","NMS_CHEMss",
    "TotalRawScoreTRUE","PartIRawScoreTRUE","PartIIRawScoreTRUE",
    "Raw_Verbal","Raw_InductiveReasoning","Raw_Quantitative","Raw_PerceptualAcuity",
    "Raw_Biology","Raw_Physics","Raw_SocialScience","Raw_Chemistry",
    "StoredRawTotal","CalculatedRawTotal_Source",
    "APT_CEM","SA_CEM","GPS_CEM","Percentile_CEM"
]
for col in num_cols:
    if col in nmat.columns:
        nmat[col] = pd.to_numeric(nmat[col], errors="coerce")

# Normalize names and keys
nmat["NAME_NORM"]   = nmat["NMA_Name"].map(normalize_name)
nmat["APPNO_CLEAN"] = nmat["NMA_AppNo"].map(clean_appno)
nmat["YEAR_INT"]    = nmat["Year"].map(clean_year)

print(f"NMAT_FINAL loaded:     {len(nmat):,} rows  |  {nmat['APPNO_CLEAN'].nunique():,} unique AppNos")
print(f"  Name coverage:       {nmat['NAME_NORM'].ne('').sum():,} non-empty names")

# ── PLE_DATA ──────────────────────────────────────────────
ple = pd.read_csv(PLE_DATA_PATH, dtype=str)
ple.columns = [c.strip() for c in ple.columns]
ple["FULL_NAME"]       = ple["FULL_NAME"].str.strip()
ple["PLE_YEAR_PASSED"] = ple["PLE_YEAR_PASSED"].map(clean_year)
ple["NAME_NORM"]       = ple["FULL_NAME"].map(normalize_name)

print(f"\nPLE_DATA loaded:       {len(ple):,} rows")
print(f"  Year range:          {int(ple['PLE_YEAR_PASSED'].min())} – {int(ple['PLE_YEAR_PASSED'].max())}")
print(f"  Unique names:        {ple['NAME_NORM'].nunique():,}")

# ── PLE_UNMATCHED ─────────────────────────────────────────
unmatched = pd.read_csv(PLE_UNMATCHED_PATH, dtype=str)
unmatched.columns = [c.strip() for c in unmatched.columns]
unmatched["FULL_NAME"]  = unmatched["FULL_NAME"].str.strip()
unmatched["NMA_AppNo"]  = unmatched["NMA_AppNo"].fillna("").str.strip()
unmatched["NAME_NORM"]  = unmatched["FULL_NAME"].map(normalize_name)
unmatched["APPNO_CLEAN"]= unmatched["NMA_AppNo"].map(clean_appno)

has_appno = unmatched["APPNO_CLEAN"].ne("").sum()
print(f"\nPLE_UNMATCHED loaded:  {len(unmatched):,} rows")
print(f"  With AppNo filled:   {has_appno:,}")
print(f"  AppNo empty:         {len(unmatched) - has_appno:,}")

# ── PLE_STILL_UNMATCHED ───────────────────────────────────
ple_still_unmatched = pd.read_csv(PLE_STILL_UNMATCHED_PATH, dtype=str)
ple_still_unmatched.columns = [c.strip() for c in ple_still_unmatched.columns]

# BUGFIX (A2/orchestrator, 2026-08): this file is a fossil of an earlier
# pipeline version that DID run fuzzy name matching -- 4,810 of its 7,207 rows
# carry MATCH_METHOD="FUZZY". Stage 2 consumes its MATCHED_APPNO values and
# labels the result DETERMINISTIC_APPNO, which laundered fuzzy provenance into
# a stage the project advertises as deterministic-only and human-auditable.
# Only rows carrying a usable MATCHED_APPNO can contribute at all, so the real
# exposure is 269 fuzzy-derived AppNos (not 4,810) -- cheap to drop, and
# dropping them makes the "no fuzzy matching" guarantee true of the lineage and
# not merely of the current source code.
if "MATCH_METHOD" in ple_still_unmatched.columns:
    _fuzzy = ple_still_unmatched["MATCH_METHOD"].astype(str).str.upper().eq("FUZZY")
    _fuzzy_with_appno = int((_fuzzy & ple_still_unmatched["MATCHED_APPNO"]
                             .fillna("").str.strip().ne("")).sum())
    ple_still_unmatched = ple_still_unmatched[~_fuzzy].copy()
    print(f"PLE_STILL_UNMATCHED: dropped {int(_fuzzy.sum()):,} FUZZY-derived rows "
          f"({_fuzzy_with_appno:,} of which carried a usable AppNo) "
          f"-- deterministic-only guarantee now holds for the lineage too")
# Determine which AppNo column is present
appno_col = "NMA_AppNo" if "NMA_AppNo" in ple_still_unmatched.columns else "MATCHED_APPNO"
if appno_col in ple_still_unmatched.columns:
    ple_still_unmatched["NMA_AppNo"] = ple_still_unmatched[appno_col].fillna("").str.strip()
else:
    ple_still_unmatched["NMA_AppNo"] = ""
ple_still_unmatched["APPNO_CLEAN"] = ple_still_unmatched["NMA_AppNo"].map(clean_appno)
ple_still_unmatched["NAME_NORM"] = ple_still_unmatched["PLE_NAME_NORM"] # Usually it's PLE_NAME_NORM
print(f"\nPLE_STILL_UNMATCHED loaded:  {len(ple_still_unmatched):,} rows")
print(f"  With AppNo filled:         {ple_still_unmatched['APPNO_CLEAN'].ne('').sum():,}")

NMAT_FINAL loaded:     178,927 rows  |  178,926 unique AppNos
  Name coverage:       178,927 non-empty names

PLE_DATA loaded:       43,630 rows
  Year range:          2011 – 2022
  Unique names:        43,630

PLE_UNMATCHED loaded:  6,600 rows
  With AppNo filled:   2,332
  AppNo empty:         4,268
PLE_STILL_UNMATCHED: dropped 4,810 FUZZY-derived rows (269 of which carried a usable AppNo) -- deterministic-only guarantee now holds for the lineage too

PLE_STILL_UNMATCHED loaded:  2,397 rows
  With AppNo filled:         52


In [6]:
# ============================================================
# CELL 5 — Build NMAT lookup structures
# ============================================================

# Group all NMAT rows by normalized name
# Each group is a list of row dicts (representing all attempts by that person)
# BUGFIX (pre-existing, found while investigating why disambiguate()'s
# Step-2 DOB filter never fired): BDATE_CLEAN must be built BEFORE
# nmat_records is captured below. nmat_records is a frozen snapshot of
# nmat's rows used to build the `candidates` lists disambiguate() reads
# via r.get("BDATE_CLEAN", ""). Previously BDATE_CLEAN was assigned to
# `nmat` AFTER this snapshot was taken, so every candidate dict lacked
# the key entirely and Step 2 (DOB/identity filter) silently fell back
# to "keep everyone" on every single call -- it was dead code.
nmat["BDATE_CLEAN"] = nmat["BDATE"].fillna("").str.strip()

# Group all NMAT rows by normalized name
# Each group is a list of row dicts (representing all attempts by that person)
nmat_records = nmat.to_dict("records")

from collections import defaultdict
nmat_by_name = defaultdict(list)
nmat_by_appno = {}

for row in nmat_records:
    name = row["NAME_NORM"]
    appno = row["APPNO_CLEAN"]
    if name:
        nmat_by_name[name].append(row)
    if appno:
        nmat_by_appno[appno] = row

nmat_name_choices = sorted(nmat_by_name.keys())

print(f"Unique NMAT names indexed:   {len(nmat_name_choices):,}")
print(f"Unique NMAT AppNos indexed:  {len(nmat_by_appno):,}")

# Person-level dedup for reporting
# Two rows = same person if same NAME_NORM + same BDATE
person_id_cols = ["NAME_NORM", "BDATE_CLEAN"]
nmat["PERSON_KEY"] = nmat["NAME_NORM"] + "||" + nmat["BDATE_CLEAN"]

# RC-3 exposure prep: log the empty-DOB rate. When BDATE_CLEAN is empty,
# PERSON_KEY degrades to name-only, which is the main mechanism behind
# name collisions merging distinct people under one PERSON_KEY.
empty_dob_rows = int((nmat["BDATE_CLEAN"] == "").sum())
empty_dob_rate = empty_dob_rows / len(nmat)
print(f"Rows with EMPTY birthdate component (PERSON_KEY degrades to name-only): "
      f"{empty_dob_rows:,} / {len(nmat):,} ({empty_dob_rate:.2%})")

unique_persons = nmat["PERSON_KEY"].nunique()
attempt_counts = nmat.groupby("PERSON_KEY")["APPNO_CLEAN"].count()

print(f"\nUnique persons (name+BDATE):  {unique_persons:,}")
print(f"Took NMAT once:               {(attempt_counts == 1).sum():,}")
print(f"Took NMAT 2+ times:           {(attempt_counts > 1).sum():,}")
print(f"Max attempts by one person:   {attempt_counts.max()}")

Unique NMAT names indexed:   123,736
Unique NMAT AppNos indexed:  178,926
Rows with EMPTY birthdate component (PERSON_KEY degrades to name-only): 25,204 / 178,927 (14.09%)

Unique persons (name+BDATE):  134,869
Took NMAT once:               101,155
Took NMAT 2+ times:           33,714
Max attempts by one person:   9


In [7]:
# ============================================================
# CELL 6 — Disambiguation helper
# ============================================================

# Funnel counters for disambiguate() (populated as it runs during Stage 1
# matching, Cell 9). Every PLE name that reaches this function starts with
# 2+ exact-name NMAT candidates; each counter is where in the funnel that
# candidate set got resolved (or didn't).
DISAMBIG_STATS = {
    "n_calls": 0,                    # PLE names with 2+ initial exact-name candidates
    "step1_no_valid_match": 0,       # ALL candidates failed the year-gap filter
    "resolved_unique": 0,            # exactly one candidate survives steps 1-3 -> FINAL_MATCH
    "rejected_ambiguous_person": 0,  # 2+ candidates still survive steps 1-3 -> rejected, not guessed
}


def disambiguate(candidates: list, ple_year: int, ple_name_norm: str) -> dict:
    """
    Apply identity-resolution rules to a list of NMAT candidate rows sharing
    one PLE record's normalized name. Returns a result dict with keys:
        selected_row, status, reason, n_before_filter, n_after_filter

    Status values:
        FINAL_MATCH               — one clean winner after steps 1-3
        AMBIGUOUS_NAME_COLLISION  — 2+ candidates still tied after steps 1-3
        NO_VALID_MATCH            — zero candidates survive the year-gap filter

    FIX (ORCHESTRATOR RULING — the most serious defect found in this pipeline):
    this used to have a Step 4 percentile-floor filter (drop, and even reject
    outright, candidates scoring below the 40th percentile) and a Step 5
    tie-break on highest NMS_PER_num. BOTH are removed. Percentile is the
    outcome variable this project studies — whether below-40th-percentile
    examinees go on to pass the PLE — so using it anywhere in IDENTITY
    resolution (a hard filter OR a tie-break) systematically prevented
    low-percentile examinees from ever being matched at all, manufacturing
    part of the very B1-B4 linkage collapse the research question is about.
    Identity is now decided ONLY by non-outcome evidence: year gap, DOB,
    latest attempt year. Exactly one survivor -> accept. More than one
    survivor -> reject rather than guess. No score-based tie-break, no coin
    flip.
    """
    n_before = len(candidates)
    DISAMBIG_STATS["n_calls"] += 1

    # ── Step 1: Year gap filter ────────────────────────────────────
    # PLE_YEAR - NMAT_YEAR >= YEAR_GAP_MIN
    gap_pass = [
        r for r in candidates
        if (
            pd.notna(r.get("YEAR_INT"))
            and (ple_year - r["YEAR_INT"]) >= YEAR_GAP_MIN
        )
    ]

    if not gap_pass:
        DISAMBIG_STATS["step1_no_valid_match"] += 1
        return {
            "selected_row": None,
            "status": "NO_VALID_MATCH",
            "reason": f"All {n_before} candidates failed year gap filter (gap < {YEAR_GAP_MIN})",
            "n_before_filter": n_before,
            "n_after_filter": 0
        }

    # ── Step 2: Identity filter (DOB) ──────────────────────────────
    # Keep records that share the most common BDATE_CLEAN. If DOB is missing
    # for all survivors → skip this step (nothing to filter on).
    has_dob = [r for r in gap_pass if r.get("BDATE_CLEAN", "") not in ("", "00/00/0000", "nan")]

    if has_dob:
        from collections import Counter
        dob_counts = Counter(r.get("BDATE_CLEAN", "") for r in has_dob)
        modal_dob = dob_counts.most_common(1)[0][0]
        identity_pass = [r for r in gap_pass if r.get("BDATE_CLEAN", "") == modal_dob]
        if not identity_pass:
            identity_pass = gap_pass  # fall back if no DOB consensus
    else:
        identity_pass = gap_pass  # no DOB available → keep all

    # ── Step 3: Take latest NMAT year ──────────────────────────────
    max_year = max(
        r["YEAR_INT"] for r in identity_pass if pd.notna(r.get("YEAR_INT"))
    )
    latest_pass = [r for r in identity_pass if r.get("YEAR_INT") == max_year]

    # ── Step 4: Final verdict (the old percentile-floor Step 4 and the
    # percentile-tiebreak Step 5 are both deleted — see docstring) ───
    if len(latest_pass) == 1:
        DISAMBIG_STATS["resolved_unique"] += 1
        return {
            "selected_row": latest_pass[0],
            "status": "FINAL_MATCH",
            "reason": f"1 candidate after year-gap + DOB + latest-year filters (gap≥{YEAR_GAP_MIN})",
            "n_before_filter": n_before,
            "n_after_filter": 1
        }

    DISAMBIG_STATS["rejected_ambiguous_person"] += 1
    return {
        "selected_row": None,
        "status": "AMBIGUOUS_NAME_COLLISION",
        "reason": f"{len(latest_pass)} candidates remain after year-gap + DOB + latest-year filters "
                  f"— no non-outcome evidence left to separate them, rejected rather than guessed",
        "n_before_filter": n_before,
        "n_after_filter": len(latest_pass)
    }


print("Disambiguation helper defined.")

Disambiguation helper defined.


In [8]:
# ============================================================
# CELL 7 — Stage 0: PLE_UNMATCHED with pre-filled AppNo
# ============================================================

print("=" * 60)
print("STAGE 0: Manual AppNo matches from PLE_UNMATCHED.csv")
print("=" * 60)

unmatched_with_appno = unmatched[unmatched["APPNO_CLEAN"].ne("")].copy()
unmatched_no_appno   = unmatched[unmatched["APPNO_CLEAN"].eq("") ].copy()

print(f"PLE_UNMATCHED rows WITH AppNo:    {len(unmatched_with_appno):,}")
print(f"PLE_UNMATCHED rows WITHOUT AppNo: {len(unmatched_no_appno):,}")

# Join on AppNo to NMAT
stage0_results = []

for _, row in unmatched_with_appno.iterrows():
    appno = row["APPNO_CLEAN"]
    nmat_row = nmat_by_appno.get(appno)

    if nmat_row is not None:
        stage0_results.append({
            "PLE_FULL_NAME":       row["FULL_NAME"],
            "PLE_NAME_NORM":       row["NAME_NORM"],
            "PLE_YEAR_PASSED":     np.nan,      # not in PLE_UNMATCHED (no year column)
            "MATCHED_APPNO":       appno,
            "MATCHED_NMA_Name":    nmat_row.get("NMA_Name"),
            "MATCHED_YEAR_INT":    nmat_row.get("YEAR_INT"),
            "MATCHED_NMS_PER_num": nmat_row.get("NMS_PER_num"),
            "YEAR_GAP":            np.nan,
            "MATCH_METHOD":        "MANUAL_APPNO_MATCH",
            "MATCH_STATUS":        "FINAL_MATCH",
            "MATCH_CONFIDENCE":    100,
            "MATCH_REASON":        "Pre-filled AppNo from PLE_UNMATCHED.csv — direct join"
        })
    else:
        stage0_results.append({
            "PLE_FULL_NAME":       row["FULL_NAME"],
            "PLE_NAME_NORM":       row["NAME_NORM"],
            "PLE_YEAR_PASSED":     np.nan,
            "MATCHED_APPNO":       appno,
            "MATCHED_NMA_Name":    None,
            "MATCHED_YEAR_INT":    np.nan,
            "MATCHED_NMS_PER_num": np.nan,
            "YEAR_GAP":            np.nan,
            "MATCH_METHOD":        "MANUAL_APPNO_MATCH",
            "MATCH_STATUS":        "APPNO_NOT_IN_NMAT",   # AppNo given but not found in NMAT_FINAL
            "MATCH_CONFIDENCE":    0,
            "MATCH_REASON":        "AppNo from PLE_UNMATCHED not found in NMAT_FINAL"
        })

stage0_df = pd.DataFrame(stage0_results)
found_s0   = (stage0_df["MATCH_STATUS"] == "FINAL_MATCH").sum()
missing_s0 = (stage0_df["MATCH_STATUS"] != "FINAL_MATCH").sum()

print(f"\nStage 0 results:")
print(f"  Matched via AppNo:    {found_s0:,}")
print(f"  AppNo not in NMAT:    {missing_s0:,}")

# The remaining unmatched (no AppNo) go to Stage 3 name matching
ple_unmatched_to_name_match = unmatched_no_appno.copy()
print(f"\n  Proceeding to name matching: {len(ple_unmatched_to_name_match):,} records")

STAGE 0: Manual AppNo matches from PLE_UNMATCHED.csv
PLE_UNMATCHED rows WITH AppNo:    2,332
PLE_UNMATCHED rows WITHOUT AppNo: 4,268



Stage 0 results:
  Matched via AppNo:    2,331
  AppNo not in NMAT:    1

  Proceeding to name matching: 4,268 records


In [9]:
# ============================================================
# CELL 8 — Build master PLE list with source tags
# ============================================================

# PLE_DATA: all 43,630 passers
# PLE_UNMATCHED (no AppNo): additional 6,600 that failed prior match

ple_all = ple[["FULL_NAME", "PLE_YEAR_PASSED", "NAME_NORM"]].copy()
ple_all["SOURCE"] = "PLE_DATA"

# For PLE_UNMATCHED with no AppNo: they are a SUBSET of PLE_DATA
# We should not double-count them.
# The PLE_UNMATCHED file represents PLE_DATA records that failed prior exact match.
# We'll attempt to match all PLE_DATA records via our own algorithm,
# so PLE_UNMATCHED without AppNo is implicitly covered.
# Stage 0 already handled the ones WITH AppNo separately.

print(f"PLE_DATA records to match (all): {len(ple_all):,}")
print(f"Already resolved in Stage 0:     {found_s0:,}  (these will be excluded from name-match stages)")

# Names already resolved in Stage 0
stage0_resolved_names = set(
    stage0_df[stage0_df["MATCH_STATUS"] == "FINAL_MATCH"]["PLE_NAME_NORM"].tolist()
)

# For name matching we process all PLE_DATA; Stage 0 results take priority
print(f"Names pre-resolved by Stage 0:   {len(stage0_resolved_names):,}")

PLE_DATA records to match (all): 43,630
Already resolved in Stage 0:     2,331  (these will be excluded from name-match stages)
Names pre-resolved by Stage 0:   2,330


In [10]:
# ============================================================
# CELL 9 — Stage 1: Exact name match on all PLE_DATA
# ============================================================

print("=" * 60)
print("STAGE 1: Exact name match — PLE_DATA vs NMAT_FINAL")
print("=" * 60)

exact_match_results = []

for _, ple_row in tqdm(ple_all.iterrows(), total=len(ple_all), desc="Exact matching"):
    pname      = ple_row["NAME_NORM"]
    ple_year   = ple_row["PLE_YEAR_PASSED"]
    full_name  = ple_row["FULL_NAME"]

    # Skip if already resolved by Stage 0
    if pname in stage0_resolved_names:
        continue

    candidates = nmat_by_name.get(pname, [])

    if len(candidates) == 0:
        # No exact match → goes to Stage 2 (fuzzy)
        exact_match_results.append({
            "PLE_FULL_NAME":       full_name,
            "PLE_NAME_NORM":       pname,
            "PLE_YEAR_PASSED":     ple_year,
            "MATCHED_APPNO":       None,
            "MATCHED_NMA_Name":    None,
            "MATCHED_YEAR_INT":    np.nan,
            "MATCHED_NMS_PER_num": np.nan,
            "YEAR_GAP":            np.nan,
            "MATCH_METHOD":        "EXACT",
            "MATCH_STATUS":        "NEEDS_FUZZY",
            "MATCH_CONFIDENCE":    0,
            "MATCH_REASON":        "No exact name match found"
        })

    elif len(candidates) == 1:
        # Single exact match — still apply year gap check
        r = candidates[0]
        year_int = r.get("YEAR_INT")
        gap = (ple_year - year_int) if (pd.notna(ple_year) and pd.notna(year_int)) else np.nan

        if pd.notna(gap) and gap < YEAR_GAP_MIN:
            exact_match_results.append({
                "PLE_FULL_NAME":       full_name,
                "PLE_NAME_NORM":       pname,
                "PLE_YEAR_PASSED":     ple_year,
                "MATCHED_APPNO":       r.get("APPNO_CLEAN"),
                "MATCHED_NMA_Name":    r.get("NMA_Name"),
                "MATCHED_YEAR_INT":    year_int,
                "MATCHED_NMS_PER_num": r.get("NMS_PER_num"),
                "YEAR_GAP":            gap,
                "MATCH_METHOD":        "EXACT",
                "MATCH_STATUS":        "NO_VALID_MATCH",
                "MATCH_CONFIDENCE":    95,
                "MATCH_REASON":        f"Single exact match but year gap {gap:.0f} < {YEAR_GAP_MIN}"
            })
        else:
            exact_match_results.append({
                "PLE_FULL_NAME":       full_name,
                "PLE_NAME_NORM":       pname,
                "PLE_YEAR_PASSED":     ple_year,
                "MATCHED_APPNO":       r.get("APPNO_CLEAN"),
                "MATCHED_NMA_Name":    r.get("NMA_Name"),
                "MATCHED_YEAR_INT":    year_int,
                "MATCHED_NMS_PER_num": r.get("NMS_PER_num"),
                "YEAR_GAP":            gap,
                "MATCH_METHOD":        "EXACT",
                "MATCH_STATUS":        "FINAL_MATCH",
                "MATCH_CONFIDENCE":    100,
                "MATCH_REASON":        "Single exact name match, year gap OK"
            })

    else:
        # Multiple exact matches → disambiguate
        result = disambiguate(candidates, int(ple_year) if pd.notna(ple_year) else 9999, pname)
        sel    = result["selected_row"]
        status = result["status"]
        year_int = sel.get("YEAR_INT") if sel else np.nan
        gap = (ple_year - year_int) if (pd.notna(ple_year) and pd.notna(year_int)) else np.nan

        exact_match_results.append({
            "PLE_FULL_NAME":       full_name,
            "PLE_NAME_NORM":       pname,
            "PLE_YEAR_PASSED":     ple_year,
            "MATCHED_APPNO":       sel.get("APPNO_CLEAN") if sel else None,
            "MATCHED_NMA_Name":    sel.get("NMA_Name") if sel else None,
            "MATCHED_YEAR_INT":    year_int,
            "MATCHED_NMS_PER_num": sel.get("NMS_PER_num") if sel else np.nan,
            "YEAR_GAP":            gap,
            "MATCH_METHOD":        "EXACT",
            "MATCH_STATUS":        status,
            "MATCH_CONFIDENCE":    100 if status == "FINAL_MATCH" else 50,
            "MATCH_REASON":        result["reason"]
        })

exact_df = pd.DataFrame(exact_match_results)

print(f"\nExact match results:")
print(exact_df["MATCH_STATUS"].value_counts(dropna=False).to_string())
print(f"\nTotal processed: {len(exact_df):,}")

print(f"\nDisambiguator funnel (candidates reaching disambiguate(), i.e. 2+ exact-name matches):")
print(f"  PLE names entering disambiguate():                      {DISAMBIG_STATS['n_calls']:,}")
print(f"  Rejected at Step 1 (year-gap, ALL candidates fail):     {DISAMBIG_STATS['step1_no_valid_match']:,}")
print(f"  Resolved to exactly one candidate (Steps 1-3):          {DISAMBIG_STATS['resolved_unique']:,}")
print(f"  Still 2+ after Steps 1-3 (AMBIGUOUS_NAME_COLLISION):    {DISAMBIG_STATS['rejected_ambiguous_person']:,}")
_funnel_check = (DISAMBIG_STATS['step1_no_valid_match'] + DISAMBIG_STATS['resolved_unique']
                  + DISAMBIG_STATS['rejected_ambiguous_person'])
print(f"  (sanity check: sum of the three outcomes above = {_funnel_check:,}, "
      f"n_calls = {DISAMBIG_STATS['n_calls']:,})")

STAGE 1: Exact name match — PLE_DATA vs NMAT_FINAL


Exact matching:   0%|          | 0/43630 [00:00<?, ?it/s]

Exact matching:   7%|▋         | 3259/43630 [00:00<00:01, 32581.80it/s]

Exact matching:  15%|█▌        | 6690/43630 [00:00<00:01, 33595.10it/s]

Exact matching:  24%|██▍       | 10546/43630 [00:00<00:00, 35860.00it/s]

...

Exact matching:  79%|███████▉  | 34521/43630 [00:00<00:00, 40141.22it/s]

Exact matching:  89%|████████▉ | 38755/43630 [00:01<00:00, 40817.36it/s]

Exact matching:  99%|█████████▉| 43326/43630 [00:01<00:00, 42312.36it/s]

Exact matching: 100%|██████████| 43630/43630 [00:01<00:00, 39373.04it/s]


Exact match results:
MATCH_STATUS
FINAL_MATCH                 33461
NEEDS_FUZZY                  4208
AMBIGUOUS_NAME_COLLISION     3578
NO_VALID_MATCH                 53

Total processed: 41,300

Disambiguator funnel (candidates reaching disambiguate(), i.e. 2+ exact-name matches):
  PLE names entering disambiguate():                      13,895
  Rejected at Step 1 (year-gap, ALL candidates fail):     1
  Resolved to exactly one candidate (Steps 1-3):          10,316
  Still 2+ after Steps 1-3 (AMBIGUOUS_NAME_COLLISION):    3,578
  (sanity check: sum of the three outcomes above = 13,895, n_calls = 13,895)


In [11]:
# ============================================================
# CELL 10 — Stage 2: Deterministic AppNo Match
# ============================================================

print("=" * 60)
print("STAGE 2: Deterministic AppNo Match")
print("=" * 60)

deterministic_results = []
has_appno_records = ple_still_unmatched[ple_still_unmatched["APPNO_CLEAN"].ne("")]
no_appno_records = ple_still_unmatched[ple_still_unmatched["APPNO_CLEAN"].eq("")]

print(f"Records with AppNo to match deterministically: {len(has_appno_records):,}")
print(f"Records missing AppNo:                       {len(no_appno_records):,}")

for _, row in has_appno_records.iterrows():
    appno = row["APPNO_CLEAN"]
    nmat_row = nmat_by_appno.get(appno)
    if nmat_row is not None:
        deterministic_results.append({
            "PLE_FULL_NAME":       row["PLE_FULL_NAME"],
            "PLE_NAME_NORM":       row["NAME_NORM"],
            "PLE_YEAR_PASSED":     float(row["PLE_YEAR_PASSED"]) if pd.notna(row["PLE_YEAR_PASSED"]) and str(row["PLE_YEAR_PASSED"]).strip() != '' else np.nan,
            "MATCHED_APPNO":       appno,
            "MATCHED_NMA_Name":    nmat_row.get("NMA_Name"),
            "MATCHED_YEAR_INT":    nmat_row.get("YEAR_INT"),
            "MATCHED_NMS_PER_num": nmat_row.get("NMS_PER_num"),
            "YEAR_GAP":            np.nan, # or calculate it if needed
            "MATCH_METHOD":        "DETERMINISTIC_APPNO",
            "MATCH_STATUS":        "FINAL_MATCH",
            "MATCH_CONFIDENCE":    100,
            "MATCH_REASON":        "Matched deterministically via provided NMA_AppNo"
        })
    else:
        deterministic_results.append({
            "PLE_FULL_NAME":       row["PLE_FULL_NAME"],
            "PLE_NAME_NORM":       row["NAME_NORM"],
            "PLE_YEAR_PASSED":     float(row["PLE_YEAR_PASSED"]) if pd.notna(row["PLE_YEAR_PASSED"]) and str(row["PLE_YEAR_PASSED"]).strip() != '' else np.nan,
            "MATCHED_APPNO":       appno,
            "MATCHED_NMA_Name":    None,
            "MATCHED_YEAR_INT":    np.nan,
            "MATCHED_NMS_PER_num": np.nan,
            "YEAR_GAP":            np.nan,
            "MATCH_METHOD":        "DETERMINISTIC_APPNO",
            "MATCH_STATUS":        "APPNO_NOT_IN_NMAT",
            "MATCH_CONFIDENCE":    0,
            "MATCH_REASON":        "Provided NMA_AppNo not found in NMAT_FINAL"
        })

for _, row in no_appno_records.iterrows():
    deterministic_results.append({
        "PLE_FULL_NAME":       row["PLE_FULL_NAME"],
        "PLE_NAME_NORM":       row["NAME_NORM"],
        "PLE_YEAR_PASSED":     float(row["PLE_YEAR_PASSED"]) if pd.notna(row["PLE_YEAR_PASSED"]) and str(row["PLE_YEAR_PASSED"]).strip() != '' else np.nan,
        "MATCHED_APPNO":       None,
        "MATCHED_NMA_Name":    None,
        "MATCHED_YEAR_INT":    np.nan,
        "MATCHED_NMS_PER_num": np.nan,
        "YEAR_GAP":            np.nan,
        "MATCH_METHOD":        "DETERMINISTIC_APPNO",
        "MATCH_STATUS":        "UNMATCHED_NO_APPNO",
        "MATCH_CONFIDENCE":    0,
        "MATCH_REASON":        "Empty NMA_AppNo and no determinable CEM data link"
    })

deterministic_df = pd.DataFrame(deterministic_results)
if len(deterministic_df) > 0:
    print(f"\nDeterministic match results:")
    print(deterministic_df["MATCH_STATUS"].value_counts(dropna=False).to_string())
else:
    print("\nNo deterministic match results.")

STAGE 2: Deterministic AppNo Match
Records with AppNo to match deterministically: 52
Records missing AppNo:                       2,345

Deterministic match results:
MATCH_STATUS
UNMATCHED_NO_APPNO    2345
FINAL_MATCH             52


In [12]:
# ============================================================
# CELL 11 — Combine all match results into master match table
# ============================================================

print("=" * 60)
print("Combining all match stages into master match table")
print("=" * 60)

# Stage 0 — manual AppNo matches
stage0_final = stage0_df.copy()
stage0_final["SOURCE_STAGE"] = "STAGE0_MANUAL_APPNO"

# Stage 1 — exact matches (exclude NEEDS_FUZZY rows, those went to fuzzy)
exact_final = exact_df[exact_df["MATCH_STATUS"] != "NEEDS_FUZZY"].copy()
exact_final["SOURCE_STAGE"] = "STAGE1_EXACT"

# Stage 2 — deterministic results
deterministic_final = deterministic_df.copy()
deterministic_final["SOURCE_STAGE"] = "STAGE2_DETERMINISTIC_APPNO"

# Standard columns across all stages
std_cols = [
    "PLE_FULL_NAME", "PLE_NAME_NORM", "PLE_YEAR_PASSED",
    "MATCHED_APPNO", "MATCHED_NMA_Name", "MATCHED_YEAR_INT",
    "MATCHED_NMS_PER_num", "YEAR_GAP",
    "MATCH_METHOD", "MATCH_STATUS", "MATCH_CONFIDENCE", "MATCH_REASON",
    "SOURCE_STAGE"
]

# Ensure all stages have all columns
for df in [stage0_final, exact_final, deterministic_final]:
    for col in std_cols:
        if col not in df.columns:
            df[col] = np.nan

master_match = pd.concat(
    [
        stage0_final[std_cols],
        exact_final[std_cols],
        deterministic_final[std_cols]
    ],
    ignore_index=True
)

# De-duplicate: if same PLE_NAME_NORM resolved by multiple stages,
# keep the highest confidence / best status result
# Priority: FINAL_MATCH > MANUAL_APPNO_MATCH > DETERMINISTIC_APPNO > AMBIGUOUS > ...
status_priority = {
    "FINAL_MATCH": 1,
    "MANUAL_APPNO_MATCH": 1,
    "DETERMINISTIC_APPNO": 1,
    "AMBIGUOUS": 3,
    "AMBIGUOUS_NAME_COLLISION": 3,   # rejected: 2+ candidates survive steps 1-3, no non-outcome evidence to separate them
    "NO_VALID_MATCH": 5,
    "UNMATCHED_FINAL": 6,
    "APPNO_NOT_IN_NMAT": 6,
    "UNMATCHED_NO_APPNO": 6,
    "NEEDS_FUZZY": 99
}

master_match["STATUS_RANK"] = master_match["MATCH_STATUS"].map(
    lambda s: status_priority.get(str(s), 99)
)

master_match = (
    master_match
    .sort_values(["PLE_NAME_NORM", "STATUS_RANK", "MATCH_CONFIDENCE"],
                 ascending=[True, True, False])
    .drop_duplicates(subset=["PLE_NAME_NORM"], keep="first")
    .drop(columns=["STATUS_RANK"])
    .reset_index(drop=True)
)

# Case-A detection (ORCHESTRATOR RULING): the only shape in this pipeline
# where WHO passed is certain but WHICH PLE record is uncertain -- one NMAT
# person (one MATCHED_APPNO) credited by 2+ DIFFERENT PLE records (distinct
# PLE_NAME_NORM entries) after the per-name dedup above. Exposed downstream
# as PLE_YEAR_UNCERTAIN in Cell 12.
_appno_counts = (
    master_match.loc[master_match["MATCHED_APPNO"].notna(), "MATCHED_APPNO"]
    .astype(str)
    .value_counts()
)
AMBIGUOUS_YEAR_APPNOS = set(_appno_counts[_appno_counts > 1].index)
print(f"\nNMAT persons (by MATCHED_APPNO) credited by 2+ distinct PLE records "
      f"(Case A -- WHO is certain, WHICH record is uncertain): {len(AMBIGUOUS_YEAR_APPNOS):,}")

master_match.to_csv(OUTDIR / "PLE_MATCH_MASTER.csv", index=False)

print(f"\nMaster match table: {len(master_match):,} PLE records")
print("\nFinal MATCH_STATUS breakdown:")
print(master_match["MATCH_STATUS"].value_counts(dropna=False).to_string())
print("\nBy METHOD:")
print(master_match["MATCH_METHOD"].value_counts(dropna=False).to_string())

Combining all match stages into master match table



NMAT persons (by MATCHED_APPNO) credited by 2+ distinct PLE records (Case A -- WHO is certain, WHICH record is uncertain): 79



Master match table: 39,423 PLE records

Final MATCH_STATUS breakdown:
MATCH_STATUS
FINAL_MATCH                 35843
AMBIGUOUS_NAME_COLLISION     3578
NO_VALID_MATCH                  1
APPNO_NOT_IN_NMAT               1

By METHOD:
MATCH_METHOD
EXACT                  37040
MANUAL_APPNO_MATCH      2331
DETERMINISTIC_APPNO       52


In [13]:
# ============================================================
# CELL 12 — Apply IS_PLE_PASSER flag to NMAT_FINAL
# ============================================================

print("=" * 60)
print("Applying PLE match flags to NMAT_FINAL → NMAT_Ultima")
print("=" * 60)

# Only accepted statuses contribute to IS_PLE_PASSER = True for clean analysis.
# NOTE (RC-1 fix): IS_PLE_ANALYSIS_SAFE has been REMOVED. It used to be defined
# with this exact same set literal, so it was a byte-identical duplicate of
# IS_PLE_PASSER rather than the documented "Year<=2014 observable cohort" it
# claimed to be. The real observable-cohort flag, IS_OBSERVABLE_COHORT =
# (Year<=2014), is built below after the merge — it is a genuinely different
# column from IS_PLE_PASSER, not a renamed alias of it.
# AMBIGUOUS_NAME_COLLISION (2+ candidates survive disambiguate()'s steps 1-3 with
# no non-outcome evidence to separate them) is deliberately NOT in this set --
# it must stay rejected, not coin-flipped.
accepted_statuses = {"FINAL_MATCH", "MANUAL_APPNO_MATCH", "DETERMINISTIC_APPNO"}

# Build AppNo → PLE match info lookup
# Only rows with a valid MATCHED_APPNO
appno_to_ple = {}
for _, mrow in master_match.iterrows():
    appno = str(mrow.get("MATCHED_APPNO", "") or "").strip()
    if appno and appno != "nan":
        appno_to_ple[appno] = mrow.to_dict()

# Name-based lookup, RESTRICTED to match rows that carry NO MATCHED_APPNO.
#
# BUGFIX (A2/orchestrator, 2026-08): this dict was previously built from EVERY
# master_match row. Because get_ple_info() falls back to it whenever a row's own
# APPNO_CLEAN is absent from appno_to_ple, an accepted match adjudicated by
# disambiguate() to ONE specific NMAT record was then re-applied to every OTHER
# NMAT record sharing that normalized name -- silently undoing the very
# disambiguation the pipeline had just performed. Measured effect: 1,624 names
# where 2+ distinct PERSON_KEYs were both flagged IS_PLE_PASSER, covering 3,256
# people, 232 of them with genuinely different non-empty birthdates. Since
# PLE_DATA.csv holds unique names, that is one passer double-credited.
#
# Restricting to MATCHED_APPNO-less rows is safe and preserves the documented
# intent ("same name matched but no specific AppNo assigned"): of the 7,714 such
# rows, ZERO carry an accepted status, so this lookup can no longer set
# IS_PLE_PASSER=True. It only carries diagnostic metadata (AMBIGUOUS_NAME_
# COLLISION, UNMATCHED_NO_APPNO) onto name-only rows, which is what it was for.
name_to_ple = {}
_name_only_rows = 0
for _, mrow in master_match.iterrows():
    _ap = str(mrow.get("MATCHED_APPNO", "") or "").strip()
    if _ap and _ap.lower() != "nan":
        continue                      # adjudicated to a specific AppNo -- not name-scoped
    name = str(mrow.get("PLE_NAME_NORM", "") or "").strip()
    if name:
        name_to_ple[name] = mrow.to_dict()
        _name_only_rows += 1
print(f"name_to_ple built from {_name_only_rows:,} MATCHED_APPNO-less rows "
      f"(was: all {len(master_match):,} rows -- the double-crediting bug)")
assert not any(
    str(v.get("MATCH_STATUS", "")) in accepted_statuses for v in name_to_ple.values()
), "name_to_ple must never carry an accepted status -- it would re-introduce double-crediting"

# Apply to NMAT_FINAL row by row
# An NMAT row is "PLE matched" if:
#   1. Its APPNO_CLEAN appears in appno_to_ple, OR
#   2. Its NAME_NORM appears in name_to_ple AND no specific AppNo was matched
#      (i.e., MATCHED_APPNO is null in match table — handles name-only matches)

def get_ple_info(nmat_row: dict) -> dict:
    appno = nmat_row.get("APPNO_CLEAN", "")
    name  = nmat_row.get("NAME_NORM", "")

    info = appno_to_ple.get(appno)
    if info is None:
        info = name_to_ple.get(name)

    if info is None:
        return {
            "PLE_MATCH_STATUS":     "NOT_IN_PLE",
            "PLE_MATCH_METHOD":     None,
            "PLE_YEAR_PASSED":      np.nan,
            "PLE_YEAR_GAP":         np.nan,
            "PLE_MATCH_CONFIDENCE": np.nan,
            "PLE_MATCH_REASON":     None,
            "IS_PLE_PASSER":        False,
            "PLE_MATCH_OUTCOME":    "no_match",
            "PLE_YEAR_UNCERTAIN":   False,
        }

    status = str(info.get("MATCH_STATUS", ""))
    is_passer = status in accepted_statuses

    # PLE_MATCH_OUTCOME makes the metadata self-explaining: PLE_MATCH_METHOD
    # and PLE_YEAR_PASSED are populated for ANY candidate match regardless of
    # adjudication outcome, while IS_PLE_PASSER stays gated on accepted status
    # only. AMBIGUOUS_NAME_COLLISION gets its own outcome label rather than
    # the generic "rejected" because it is a specific, higher-stakes failure
    # mode (Case B name collision), not a garden-variety year-gap/percentile
    # rejection.
    if status == "AMBIGUOUS_NAME_COLLISION":
        outcome = "rejected_ambiguous_person"
    elif is_passer:
        outcome = "accepted"
    else:
        outcome = "rejected"

    return {
        "PLE_MATCH_STATUS":     status,
        "PLE_MATCH_METHOD":     info.get("MATCH_METHOD"),
        "PLE_YEAR_PASSED":      info.get("PLE_YEAR_PASSED"),
        "PLE_YEAR_GAP":         info.get("YEAR_GAP"),
        "PLE_MATCH_CONFIDENCE": info.get("MATCH_CONFIDENCE"),
        "PLE_MATCH_REASON":     info.get("MATCH_REASON"),
        "IS_PLE_PASSER":        is_passer,
        "PLE_MATCH_OUTCOME":    outcome,
        # True only for Case A DOB-confirmed tie-break accepts: WHO passed is
        # certain, but WHICH exact attempt-row/year was matched is not.
        # True only when this row's underlying PLE match is one NMAT person
        # (MATCHED_APPNO) credited by 2+ distinct PLE records: WHO passed is
        # certain, WHICH record/year is not (see AMBIGUOUS_YEAR_APPNOS, Cell 11).
        "PLE_YEAR_UNCERTAIN":   is_passer and (str(info.get("MATCHED_APPNO")) in AMBIGUOUS_YEAR_APPNOS),
    }

# Apply to all rows
ple_flags = [get_ple_info(r) for r in tqdm(nmat.to_dict("records"), desc="Applying PLE flags")]
ple_flags_df = pd.DataFrame(ple_flags)

# Merge back
nmat_ultima = pd.concat(
    [nmat.reset_index(drop=True), ple_flags_df.reset_index(drop=True)],
    axis=1
)

# Real observable-cohort flag (RC-1 fix). Year<=2014 allows >=5 years for the
# PLE to plausibly have happened by the time PLE_DATA.csv was captured (2022).
# This is deliberately NOT derived from IS_PLE_PASSER/PLE_MATCH_STATUS in any
# way, so it cannot become a tautological duplicate again.
nmat_ultima["IS_OBSERVABLE_COHORT"] = nmat_ultima["Year"] <= 2014

# ------------------------------------------------------------------
# Person-level propagation of the PLE result (replaces the old, buggy
# name-based spreading removed above).
#
# IS_PLE_PASSER is a property of a PERSON, not of one sitting: a repeat taker
# who is matched on their 2007 attempt is still a passer on their 2009 attempt,
# and the dashboards' person-level cohorts (IS_BEST_NMAT_RECORD /
# IS_BEST_OBSERVABLE_RECORD) may select whichever sitting was NOT the matched
# one. The old code achieved this accidentally via the name fallback, which
# also pooled genuinely different people who happened to share a name.
#
# Propagating on PERSON_KEY (normalized name + birthdate) instead keeps the
# person-level semantics while separating same-named people who have different
# recorded birthdates. Where the birthdate is empty (14.1% of rows) PERSON_KEY
# degrades to name-only and those people can still pool -- that is the known,
# documented PERSON_KEY weakness (see PERSON_KEY_AMBIGUOUS), not new behaviour.
_matched_rows = nmat_ultima["IS_PLE_PASSER"] == True
_passer_keys = set(nmat_ultima.loc[_matched_rows, "PERSON_KEY"])
_to_fill = nmat_ultima["PERSON_KEY"].isin(_passer_keys) & (~_matched_rows)

# Carry the matched sitting's PLE metadata onto that person's other sittings so
# person-level exposure/year-gap calculations work regardless of which sitting
# the best-record flag selects.
_src = (nmat_ultima.loc[_matched_rows, ["PERSON_KEY", "PLE_YEAR_PASSED",
                                        "PLE_MATCH_METHOD", "PLE_MATCH_STATUS"]]
        .drop_duplicates(subset=["PERSON_KEY"], keep="first")
        .set_index("PERSON_KEY"))
for _col in ["PLE_YEAR_PASSED", "PLE_MATCH_METHOD", "PLE_MATCH_STATUS"]:
    nmat_ultima.loc[_to_fill, _col] = nmat_ultima.loc[_to_fill, "PERSON_KEY"].map(_src[_col])
nmat_ultima.loc[_to_fill, "IS_PLE_PASSER"] = True
nmat_ultima.loc[_to_fill, "PLE_MATCH_OUTCOME"] = "accepted"
nmat_ultima["PLE_YEAR_GAP"] = nmat_ultima["PLE_YEAR_PASSED"] - nmat_ultima["Year"]

print("")
print(f"Person-level propagation: {int(_matched_rows.sum()):,} directly matched sittings "
      f"-> {len(_passer_keys):,} distinct passer PERSON_KEYs "
      f"-> {int(_to_fill.sum()):,} sibling sittings also flagged "
      f"({int((nmat_ultima['IS_PLE_PASSER'] == True).sum()):,} total).")

print(f"\nNMAT_Ultima shape: {nmat_ultima.shape}")
print(f"\nIS_PLE_PASSER distribution:")
print(nmat_ultima["IS_PLE_PASSER"].value_counts(dropna=False).to_string())
print(f"\nPLE_MATCH_STATUS distribution:")
print(nmat_ultima["PLE_MATCH_STATUS"].value_counts(dropna=False).to_string())
print(f"\nPLE_MATCH_OUTCOME distribution:")
print(nmat_ultima["PLE_MATCH_OUTCOME"].value_counts(dropna=False).to_string())
print(f"\nIS_OBSERVABLE_COHORT distribution:")
print(nmat_ultima["IS_OBSERVABLE_COHORT"].value_counts(dropna=False).to_string())

print(f"\nPLE_YEAR_UNCERTAIN distribution:")
print(nmat_ultima["PLE_YEAR_UNCERTAIN"].value_counts(dropna=False).to_string())
print(f"\nPLE_YEAR_UNCERTAIN = True count: {int(nmat_ultima['PLE_YEAR_UNCERTAIN'].sum()):,} rows "
      f"(one NMAT person credited by 2+ distinct PLE records; "
      f"{len(AMBIGUOUS_YEAR_APPNOS):,} such MATCHED_APPNO values found in Cell 11)")

assert not (nmat_ultima["IS_OBSERVABLE_COHORT"] == nmat_ultima["IS_PLE_PASSER"]).all(), (
    "IS_OBSERVABLE_COHORT must not be a tautological duplicate of IS_PLE_PASSER (RC-1 regression!)"
)
print("\nAssertion OK: IS_OBSERVABLE_COHORT is NOT a duplicate of IS_PLE_PASSER.")

Applying PLE match flags to NMAT_FINAL → NMAT_Ultima


name_to_ple built from 3,579 MATCHED_APPNO-less rows (was: all 39,423 rows -- the double-crediting bug)


Applying PLE flags:   0%|          | 0/178927 [00:00<?, ?it/s]

Applying PLE flags:  19%|█▉        | 34672/178927 [00:00<00:02, 70829.63it/s]

Applying PLE flags:  53%|█████▎    | 94674/178927 [00:00<00:00, 190375.64it/s]

Applying PLE flags:  91%|█████████ | 161934/178927 [00:00<00:00, 308027.74it/s]

Applying PLE flags: 100%|██████████| 178927/178927 [00:00<00:00, 251156.84it/s]


Person-level propagation: 35,764 directly matched sittings -> 35,746 distinct passer PERSON_KEYs -> 11,721 sibling sittings also flagged (47,485 total).

NMAT_Ultima shape: (178927, 116)

IS_PLE_PASSER distribution:
IS_PLE_PASSER
False    131442
True      47485

PLE_MATCH_STATUS distribution:
PLE_MATCH_STATUS
NOT_IN_PLE                  123233
FINAL_MATCH                  47485
AMBIGUOUS_NAME_COLLISION      8207
NO_VALID_MATCH                   2

PLE_MATCH_OUTCOME distribution:
PLE_MATCH_OUTCOME
no_match                     123233
accepted                      47485
rejected_ambiguous_person      8207
rejected                          2

IS_OBSERVABLE_COHORT distribution:
IS_OBSERVABLE_COHORT
False    90783
True     88144

PLE_YEAR_UNCERTAIN distribution:
PLE_YEAR_UNCERTAIN
False    178848
True         79

PLE_YEAR_UNCERTAIN = True count: 79 rows (one NMAT person credited by 2+ distinct PLE records; 79 such MATCHED_APPNO values found in Cell 11)

Assertion OK: IS_OBSERVABLE_COHORT is

In [14]:
# ============================================================
# CELL 13 — IS_BEST_NMAT_RECORD, IS_BEST_OBSERVABLE_RECORD, PERSON_KEY_AMBIGUOUS
# ============================================================
# RC-2 fix: ONE uniform selection rule for EVERY person, passers and
# non-passers alike (the old rule picked passers' PLE-matched attempt but
# everyone else's highest-percentile attempt — two different rules for the
# two groups later being compared, and it silently zero-flagged 1,311 people
# whose matched attempt could not be identified).
#     highest NMS_PER_num -> latest Year -> lowest APPNO_CLEAN
# Rows with a null NMS_PER_num still participate (na_position="last"), so
# every PERSON_KEY receives exactly one True.

print("=" * 60)
print("Flagging IS_BEST_NMAT_RECORD (uniform rule)")
print("=" * 60)

# Numeric APPNO_CLEAN so "lowest" is a real numeric tie-break, not a string
# sort (which would order "9" after "10").
nmat_ultima["_APPNO_NUM"] = pd.to_numeric(nmat_ultima["APPNO_CLEAN"], errors="coerce")

_ranked = nmat_ultima.sort_values(
    ["PERSON_KEY", "NMS_PER_num", "Year", "_APPNO_NUM"],
    ascending=[True, False, False, True],
    na_position="last",
)
best_idx = _ranked.groupby("PERSON_KEY").head(1).index

nmat_ultima["IS_BEST_NMAT_RECORD"] = False
nmat_ultima.loc[best_idx, "IS_BEST_NMAT_RECORD"] = True

# Hard invariant: exactly one True per PERSON_KEY.
_counts = nmat_ultima.groupby("PERSON_KEY")["IS_BEST_NMAT_RECORD"].sum()
assert _counts.eq(1).all(), (
    f"IS_BEST_NMAT_RECORD invariant violated: "
    f"{(~_counts.eq(1)).sum():,} PERSON_KEYs do not have exactly one True"
)
assert nmat_ultima["IS_BEST_NMAT_RECORD"].sum() == nmat_ultima["PERSON_KEY"].nunique()
print("Assertion OK: exactly one IS_BEST_NMAT_RECORD=True per PERSON_KEY.")

best_records = nmat_ultima["IS_BEST_NMAT_RECORD"].sum()
print(f"IS_BEST_NMAT_RECORD = True:  {best_records:,}  rows  (== {nmat_ultima['PERSON_KEY'].nunique():,} unique PERSON_KEYs)")
print(f"  of which IS_PLE_PASSER:    {nmat_ultima[nmat_ultima['IS_BEST_NMAT_RECORD'] & nmat_ultima['IS_PLE_PASSER']].shape[0]:,}")
print(f"  of which NOT PLE passer:   {nmat_ultima[nmat_ultima['IS_BEST_NMAT_RECORD'] & ~nmat_ultima['IS_PLE_PASSER']].shape[0]:,}")

# ------------------------------------------------------------------
# IS_BEST_OBSERVABLE_RECORD (ORCHESTRATOR RULING §2a) — the person's best
# attempt WITHIN the observable window only (Year<=2014). This is NOT the
# same as IS_BEST_NMAT_RECORD & Year<=2014: a person who sat in 2013 and
# again in 2016 has their 2016 attempt selected as overall-best, which would
# silently drop them from every observable-cohort count. This flag re-applies
# the identical tie-break chain restricted to Year<=2014 rows, so it uses
# only information available within the observable window.
# ------------------------------------------------------------------
print("\n" + "=" * 60)
print("Flagging IS_BEST_OBSERVABLE_RECORD")
print("=" * 60)

nmat_ultima["IS_BEST_OBSERVABLE_RECORD"] = False
_obs_mask = nmat_ultima["IS_OBSERVABLE_COHORT"]
_obs_ranked = nmat_ultima.loc[_obs_mask].sort_values(
    ["PERSON_KEY", "NMS_PER_num", "Year", "_APPNO_NUM"],
    ascending=[True, False, False, True],
    na_position="last",
)
best_obs_idx = _obs_ranked.groupby("PERSON_KEY").head(1).index
nmat_ultima.loc[best_obs_idx, "IS_BEST_OBSERVABLE_RECORD"] = True

_obs_counts = nmat_ultima.loc[_obs_mask].groupby("PERSON_KEY")["IS_BEST_OBSERVABLE_RECORD"].sum()
assert _obs_counts.eq(1).all(), (
    f"IS_BEST_OBSERVABLE_RECORD invariant violated: "
    f"{(~_obs_counts.eq(1)).sum():,} observable PERSON_KEYs do not have exactly one True"
)
assert nmat_ultima["IS_BEST_OBSERVABLE_RECORD"].sum() == nmat_ultima.loc[_obs_mask, "PERSON_KEY"].nunique()
print("Assertion OK: exactly one IS_BEST_OBSERVABLE_RECORD=True per observable PERSON_KEY.")

nmat_ultima = nmat_ultima.drop(columns=["_APPNO_NUM"])

n_best_obs = nmat_ultima["IS_BEST_OBSERVABLE_RECORD"].sum()
n_naive_obs = int((nmat_ultima["IS_BEST_NMAT_RECORD"] & _obs_mask).sum())
print(f"IS_BEST_OBSERVABLE_RECORD = True: {n_best_obs:,} people (unique PERSON_KEYs with any Year<=2014 row)")
print(f"Naive IS_BEST_NMAT_RECORD & Year<=2014 (WRONG, do NOT use downstream): {n_naive_obs:,} people")

# ------------------------------------------------------------------
# PERSON_KEY_AMBIGUOUS (ORCHESTRATOR RULING §2b) — True where a PERSON_KEY
# shows a contradictory SEX across its rows. SEX only, NOT university:
# repeat takers legitimately record their undergrad institution differently
# across sittings (branch naming, abbreviations), so a university
# disagreement alone is not evidence of a name collision and would flag
# thousands of keys that are not actually ambiguous. Do NOT try to repair
# PERSON_KEY here — this flag only surfaces the uncertainty (RC-3).
# ------------------------------------------------------------------
print("\n" + "=" * 60)
print("Flagging PERSON_KEY_AMBIGUOUS (contradictory SEX only)")
print("=" * 60)

_sex_nunique = nmat_ultima.groupby("PERSON_KEY")["SEX"].transform(lambda s: s.nunique(dropna=True))
nmat_ultima["PERSON_KEY_AMBIGUOUS"] = _sex_nunique > 1

n_ambiguous_keys = nmat_ultima.loc[nmat_ultima["PERSON_KEY_AMBIGUOUS"], "PERSON_KEY"].nunique()
print(f"PERSON_KEY_AMBIGUOUS keys (contradictory SEX): {n_ambiguous_keys:,}")

# Diagnostic ONLY — explicitly excluded from the flag per orchestrator ruling.
_univ_nunique = nmat_ultima.groupby("PERSON_KEY")["UNIVERSITY"].transform(lambda s: s.nunique(dropna=True))
n_univ_diff_keys = nmat_ultima.loc[_univ_nunique > 1, "PERSON_KEY"].nunique()
n_both_keys = nmat_ultima.loc[
    nmat_ultima["PERSON_KEY_AMBIGUOUS"] & (_univ_nunique > 1), "PERSON_KEY"
].nunique()
print(f"[diagnostic only, NOT in PERSON_KEY_AMBIGUOUS] Keys with >1 distinct UNIVERSITY: {n_univ_diff_keys:,}")
print(f"[diagnostic only, NOT in PERSON_KEY_AMBIGUOUS] Keys with BOTH SEX and UNIVERSITY contradictions: {n_both_keys:,}")

Flagging IS_BEST_NMAT_RECORD (uniform rule)


Assertion OK: exactly one IS_BEST_NMAT_RECORD=True per PERSON_KEY.
IS_BEST_NMAT_RECORD = True:  134,869  rows  (== 134,869 unique PERSON_KEYs)
  of which IS_PLE_PASSER:    35,746
  of which NOT PLE passer:   99,123

Flagging IS_BEST_OBSERVABLE_RECORD


Assertion OK: exactly one IS_BEST_OBSERVABLE_RECORD=True per observable PERSON_KEY.
IS_BEST_OBSERVABLE_RECORD = True: 69,503 people (unique PERSON_KEYs with any Year<=2014 row)
Naive IS_BEST_NMAT_RECORD & Year<=2014 (WRONG, do NOT use downstream): 65,782 people

Flagging PERSON_KEY_AMBIGUOUS (contradictory SEX only)


PERSON_KEY_AMBIGUOUS keys (contradictory SEX): 6,148


[diagnostic only, NOT in PERSON_KEY_AMBIGUOUS] Keys with >1 distinct UNIVERSITY: 27,053
[diagnostic only, NOT in PERSON_KEY_AMBIGUOUS] Keys with BOTH SEX and UNIVERSITY contradictions: 5,051


In [15]:
# ============================================================
# CELL 14 — Save NMAT_Ultima.csv
# ============================================================

# Reorder: PLE columns come after university columns, before score columns
ple_new_cols = [
    "IS_PLE_PASSER",
    "IS_OBSERVABLE_COHORT",
    "IS_BEST_NMAT_RECORD",
    "IS_BEST_OBSERVABLE_RECORD",
    "PERSON_KEY_AMBIGUOUS",
    "PLE_MATCH_STATUS",
    "PLE_MATCH_METHOD",
    "PLE_MATCH_OUTCOME",
    "PLE_YEAR_UNCERTAIN",
    "PLE_YEAR_PASSED",
    "PLE_YEAR_GAP",
    "PLE_MATCH_CONFIDENCE",
    "PLE_MATCH_REASON",
]

existing_cols = [c for c in nmat_ultima.columns if c not in ple_new_cols]

# Insert PLE cols after "evidence_summary" column if it exists
try:
    insert_after = existing_cols.index("evidence_summary") + 1
except ValueError:
    insert_after = 40  # fallback

final_col_order = (
    existing_cols[:insert_after]
    + ple_new_cols
    + existing_cols[insert_after:]
)

nmat_ultima = nmat_ultima[final_col_order]
nmat_ultima.to_csv(ULTIMA_PATH, index=False)
nmat_ultima.to_parquet(str(ULTIMA_PATH).replace(".csv", ".parquet"), index=False)

print(f"✅ NMAT_Ultima saved: {ULTIMA_PATH}")
print(f"   Shape: {nmat_ultima.shape}")
print(f"   Columns: {len(nmat_ultima.columns)}")

✅ NMAT_Ultima saved: dataset\NMAT_Ultima.csv
   Shape: (178927, 119)
   Columns: 119


In [16]:
# ============================================================
# CELL 15 — Comprehensive validation report
# ============================================================

print("=" * 70)
print("COMPREHENSIVE VALIDATION REPORT")
print("=" * 70)

total_ple     = len(ple)
total_nmat    = len(nmat_ultima)
total_persons = nmat_ultima["PERSON_KEY"].nunique()

# PLE matching summary
ms = master_match["MATCH_STATUS"].value_counts(dropna=False)

final_matched = ms.get("FINAL_MATCH", 0) + ms.get("MANUAL_APPNO_MATCH", 0) + ms.get("DETERMINISTIC_APPNO", 0)
ambiguous     = ms.get("AMBIGUOUS", 0)
no_valid      = ms.get("NO_VALID_MATCH", 0)
unmatched_fin = ms.get("UNMATCHED_FINAL", 0)
unmatched_no_appno = ms.get("UNMATCHED_NO_APPNO", 0)

print(f"\n{'─'*55}")
print(f"PLE PASSERS (2011–2022)")
print(f"{'─'*55}")
print(f"  Total PLE passers:                  {total_ple:>8,}")
print(f"  FINAL_MATCH (exact/manual/appno):   {final_matched:>8,}  ({final_matched/total_ple*100:.2f}%)")
print(f"  AMBIGUOUS (flagged, included):      {ambiguous:>8,}  ({ambiguous/total_ple*100:.2f}%)")
print(f"  NO_VALID_MATCH:                     {no_valid:>8,}  ({no_valid/total_ple*100:.2f}%)")
print(f"  UNMATCHED_FINAL:                    {unmatched_fin:>8,}  ({unmatched_fin/total_ple*100:.2f}%)")
print(f"  UNMATCHED_NO_APPNO:                 {unmatched_no_appno:>8,}  ({unmatched_no_appno/total_ple*100:.2f}%)")
total_accepted = final_matched
print(f"\n  Total accepted for analysis:        {total_accepted:>8,}  ({total_accepted/total_ple*100:.2f}%)")

print(f"\n{'─'*55}")
print(f"NMAT_FINAL (2006–2018)")
print(f"{'─'*55}")
print(f"  Total NMAT rows:                    {total_nmat:>8,}")
print(f"  Unique NMAT persons (name+DOB):     {total_persons:>8,}")

attempt_counts2 = nmat_ultima.groupby("PERSON_KEY")["APPNO_CLEAN"].count()
print(f"  Took NMAT once:                     {(attempt_counts2 == 1).sum():>8,}")
print(f"  Took NMAT 2+ times:                 {(attempt_counts2 > 1).sum():>8,}")
print(f"  Max attempts by one person:         {attempt_counts2.max():>8}")

print(f"\n{'─'*55}")
print(f"NMAT ROWS WITH PLE FLAG")
print(f"{'─'*55}")
print(f"  IS_PLE_PASSER = True:               {nmat_ultima['IS_PLE_PASSER'].sum():>8,}")
print(f"  IS_OBSERVABLE_COHORT = True:        {nmat_ultima['IS_OBSERVABLE_COHORT'].sum():>8,}")
print(f"  IS_BEST_OBSERVABLE_RECORD = True:   {nmat_ultima['IS_BEST_OBSERVABLE_RECORD'].sum():>8,}")
print(f"  PERSON_KEY_AMBIGUOUS keys:          {nmat_ultima.loc[nmat_ultima['PERSON_KEY_AMBIGUOUS'], 'PERSON_KEY'].nunique():>8,}")
print(f"  IS_BEST_NMAT_RECORD = True:         {nmat_ultima['IS_BEST_NMAT_RECORD'].sum():>8,}")

print(f"\n{'─'*55}")
print(f"YEAR GAP DISTRIBUTION (PLE – NMAT)")
print(f"{'─'*55}")
gap_stats = master_match[master_match["YEAR_GAP"].notna()]["YEAR_GAP"]
print(f"  Median gap:   {gap_stats.median():.1f} years")
print(f"  Mean gap:     {gap_stats.mean():.1f} years")
print(f"  Min gap:      {gap_stats.min():.0f} years")
print(f"  Max gap:      {gap_stats.max():.0f} years")
print(f"  Distribution:")
print(gap_stats.value_counts().sort_index().to_string())

print(f"\n{'─'*55}")
print(f"PLE_MATCH_STATUS IN NMAT_ULTIMA")
print(f"{'─'*55}")
print(nmat_ultima["PLE_MATCH_STATUS"].value_counts(dropna=False).to_string())

COMPREHENSIVE VALIDATION REPORT

───────────────────────────────────────────────────────
PLE PASSERS (2011–2022)
───────────────────────────────────────────────────────
  Total PLE passers:                    43,630
  FINAL_MATCH (exact/manual/appno):     35,843  (82.15%)
  AMBIGUOUS (flagged, included):             0  (0.00%)
  NO_VALID_MATCH:                            1  (0.00%)
  UNMATCHED_FINAL:                           0  (0.00%)
  UNMATCHED_NO_APPNO:                        0  (0.00%)

  Total accepted for analysis:          35,843  (82.15%)

───────────────────────────────────────────────────────
NMAT_FINAL (2006–2018)
───────────────────────────────────────────────────────
  Total NMAT rows:                     178,927
  Unique NMAT persons (name+DOB):      134,869
  Took NMAT once:                      101,155
  Took NMAT 2+ times:                   33,714
  Max attempts by one person:                9

───────────────────────────────────────────────────────
NMAT ROWS WITH PL

In [17]:
# ============================================================
# CELL 16 — Save all audit outputs
# ============================================================

master_match.to_csv(OUTDIR / "PLE_MATCH_MASTER.csv", index=False)

# Unmatched PLE passers for review
unmatched_output = master_match[
    master_match["MATCH_STATUS"].isin(["UNMATCHED_FINAL", "NO_VALID_MATCH", "UNMATCHED_NO_APPNO"])
].copy()
unmatched_output.to_csv(OUTDIR / "PLE_STILL_UNMATCHED_v2.csv", index=False) # Saving as v2 so we don't overwrite the input if they are in the same folder

# Ambiguous for manual review
ambiguous_output = master_match[master_match["MATCH_STATUS"] == "AMBIGUOUS"].copy()
ambiguous_output.to_csv(OUTDIR / "PLE_AMBIGUOUS_REVIEW.csv", index=False)

# PLE passers in NMAT (best record only)
ple_in_nmat = nmat_ultima[
    nmat_ultima["IS_PLE_PASSER"] & nmat_ultima["IS_BEST_NMAT_RECORD"]
].copy()
ple_in_nmat.to_csv(OUTDIR / "PLE_PASSERS_IN_NMAT.csv", index=False)

print("Saved audit outputs:")
print(f"  PLE_MATCH_MASTER.csv          → {len(master_match):,} rows")
print(f"  PLE_STILL_UNMATCHED_v2.csv    → {len(unmatched_output):,} rows")
print(f"  PLE_AMBIGUOUS_REVIEW.csv      → {len(ambiguous_output):,} rows")
print(f"  PLE_PASSERS_IN_NMAT.csv       → {len(ple_in_nmat):,} rows")
print(f"\n✅ NMAT_Ultima.csv is ready for all analyses.")
print(f"   Use IS_BEST_NMAT_RECORD=True for person-level analysis.")
print(f"   Use IS_OBSERVABLE_COHORT=True for the Year<=2014 observable cohort.")
print(f"   Use IS_BEST_OBSERVABLE_RECORD=True for person-level observable-cohort analysis")
print(f"   (NOT IS_BEST_NMAT_RECORD & Year<=2014 — that silently drops repeat takers whose")
print(f"   overall-best attempt falls after 2014).")
print(f"   IS_PLE_PASSER=True EXCLUDES AMBIGUOUS/NO_VALID_MATCH; it is the authoritative")
print(f"   confirmed-passer flag. See PLE_MATCH_OUTCOME for accepted/rejected metadata.")

Saved audit outputs:
  PLE_MATCH_MASTER.csv          → 39,423 rows
  PLE_STILL_UNMATCHED_v2.csv    → 1 rows
  PLE_AMBIGUOUS_REVIEW.csv      → 0 rows
  PLE_PASSERS_IN_NMAT.csv       → 35,746 rows

✅ NMAT_Ultima.csv is ready for all analyses.
   Use IS_BEST_NMAT_RECORD=True for person-level analysis.
   Use IS_OBSERVABLE_COHORT=True for the Year<=2014 observable cohort.
   Use IS_BEST_OBSERVABLE_RECORD=True for person-level observable-cohort analysis
   (NOT IS_BEST_NMAT_RECORD & Year<=2014 — that silently drops repeat takers whose
   overall-best attempt falls after 2014).
   IS_PLE_PASSER=True EXCLUDES AMBIGUOUS/NO_VALID_MATCH; it is the authoritative
   confirmed-passer flag. See PLE_MATCH_OUTCOME for accepted/rejected metadata.
